# Redrob Hackathon — Kaggle Dual-GPU Pipeline

**Goal:** Rank top 100 from 100K for Senior AI Engineer.

**Kaggle setup:** *Settings → Accelerator → GPU T4 x2*

### Pipeline
1. Clone + extract dataset
2. Install deps
3. GPU precompute: bi-encoder (DataParallel) + BM25 + cross-encoder + features
4. CPU rank: XGBoost LTR → top 100
5. Validate + download

## Step 1 — Clone + Extract

In [ ]:
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
!ls dataset/India_runs_data_and_ai_challenge/

In [ ]:
import os
CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400
print("Dataset OK")

## Step 2 — Install Dependencies + GPU Check

In [ ]:
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost

import torch
gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")
print(f"Total GPUs: {gpu_count}")

## Step 3 — Pre-compute (Dual GPU)

Bi-encoder embeddings with DataParallel + BM25 + cross-encoder re-ranking + 45+ features.

**Expected:** ~15–25 min on 2x T4.

In [2]:
import json, sys, time, os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

h:\india_runs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Ensure we're in the repo directory
if not os.path.exists('src/features.py'):
    os.chdir('india-runs')
print(f'Working dir: {os.getcwd()}')

CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
ARTIFACTS = "./artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
BATCH_SIZE = 256
CROSS_ENCODER_TOP_N = 1000

# ---- Load candidates & extract features ----
print("Loading candidates + extracting features...")
sys.path.insert(0, '.')
from src.features import extract_features
from src.bm25 import compute_bm25_scores
from src.precompute import JD_QUERY
from dataclasses import asdict

candidates = []
with open(CANDIDATES) as f:
    for line in f:
        candidates.append(json.loads(line))

feat_dicts = []
all_texts = []
for cand in tqdm(candidates, desc="Features"):
    fd = asdict(extract_features(cand))
    feat_dicts.append(fd)
    all_texts.append(fd.get("profile_text", ""))
print(f"  {len(candidates)} candidates loaded")

# ---- BM25 ----
print("Computing BM25 scores...")
t_bm25 = time.time()
bm25_scores = compute_bm25_scores(all_texts, JD_QUERY)
print(f"  BM25 done in {time.time() - t_bm25:.1f}s")

# ---- Bi-encoder embeddings (single GPU, no DataParallel) ----
# DataParallel adds ~5x overhead for SentenceTransformer encoding.
# Single GPU with large batch is much faster (~3-5s/batch vs 25s/batch).
print("Loading bi-encoder...")
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda:0")
model.max_seq_length = 256  # profiles rarely exceed 256 tokens
model.half()  # fp16 — 2x faster on T4, negligible quality loss

jd_emb = model.encode([JD_QUERY], normalize_embeddings=True, batch_size=1)
np.save(os.path.join(ARTIFACTS, "jd_embedding.npy"), jd_emb[0])

print(f"Embedding {len(all_texts)} candidates (single GPU, batch={BATCH_SIZE})...")
t0 = time.time()
embeddings = model.encode(
    all_texts,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    device="cuda:0",
)
embeddings = embeddings.astype(np.float32)
np.save(os.path.join(ARTIFACTS, "embeddings.npy"), embeddings)
print(f"Embeddings: {embeddings.shape} in {time.time() - t0:.1f}s")

# Semantic similarities
semantic_scores = embeddings @ jd_emb[0].astype(np.float32)

# ---- Cross-encoder re-ranking top-N ----
print(f"Cross-encoder re-ranking top-{CROSS_ENCODER_TOP_N}...")
del model
torch.cuda.empty_cache()

top_n_idx = np.argsort(-semantic_scores)[:CROSS_ENCODER_TOP_N]
cross_enc = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda:0")
ce_pairs = [(JD_QUERY, all_texts[idx]) for idx in top_n_idx]
ce_raw = cross_enc.predict(ce_pairs, batch_size=128, show_progress_bar=True)

ce_min, ce_max = float(ce_raw.min()), float(ce_raw.max())
ce_range = ce_max - ce_min if ce_max > ce_min else 1.0
cross_encoder_scores = np.zeros(len(all_texts), dtype=np.float32)
for j, idx in enumerate(top_n_idx):
    cross_encoder_scores[idx] = (float(ce_raw[j]) - ce_min) / ce_range

del cross_enc
torch.cuda.empty_cache()

# ---- Save features ----
feat_df = pd.DataFrame(feat_dicts)
feat_df.drop(columns=["profile_text"], inplace=True)
feat_df["bm25_score"] = bm25_scores
feat_df["cross_encoder_score"] = cross_encoder_scores.tolist()
feat_df.to_parquet(os.path.join(ARTIFACTS, "features.parquet"), index=False)
print(f"Features saved: {len(feat_df)} rows, {len(feat_df.columns)} cols")

In [3]:
# Verify
emb = np.load('artifacts/embeddings.npy')
jd = np.load('artifacts/jd_embedding.npy')
df = pd.read_parquet('artifacts/features.parquet')
print(f"embeddings: {emb.shape} | jd: {jd.shape} | features: {len(df)} rows")
print(f"bm25 range: [{df.bm25_score.min():.3f}, {df.bm25_score.max():.3f}]")
print(f"cross-enc range: [{df.cross_encoder_score.min():.3f}, {df.cross_encoder_score.max():.3f}]")
print("Artifacts OK")

embeddings: (100000, 1024) | jd: (1024,) | features: 100000 rows
bm25 range: [0.001, 1.000]
cross-enc range: [0.000, 1.000]
Artifacts OK


## Step 4 — XGBoost LTR Ranking (CPU, <3 min)

In [4]:
!python src/rank.py --artifacts ./artifacts --out ./jashwanth_s.csv

Loading precomputed artifacts...
  100000 candidates, embeddings: (100000, 1024)
Computing semantic similarities...
Computing heuristic scores...
Building LTR feature matrix...
Creating pseudo-relevance labels...
  Label distribution: {np.int32(0): 76781, np.int32(1): 18338, np.int32(2): 1454, np.int32(3): 2773, np.int32(4): 654}
Training XGBoost LTR (rank:pairwise)...
  DMatrix: 100000 samples, 41 features, group=[100000]
[0]	train-ndcg@10:0.26746
[50]	train-ndcg@10:1.00000
[100]	train-ndcg@10:1.00000
[150]	train-ndcg@10:1.00000
[199]	train-ndcg@10:1.00000
  LTR scores: min=0.0000, max=1.0000

Submission written to jashwanth_s.csv (53.3s)
  Score range: 1.0000 → 0.7680

  Top-10 LTR features by gain:
    title_relevance_tier: 6.2
    cross_encoder_score: 5.1
    semantic_sim: 4.1
    has_product_company: 4.0
    career_ai_depth_ratio: 3.0
    bm25_score: 2.6
    skills_match_score: 2.5
    heuristic_score: 2.0
    vector_search_experience: 1.5
    career_retrieval_months: 0.4


## Step 5 — Validate & Download

In [5]:
import pandas as pd, sys
from dataclasses import fields

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1))

# Tie-break check
rows = df.sort_values('rank')[['rank','score','candidate_id']].values.tolist()
for i in range(len(rows)-1):
    if rows[i][1] == rows[i+1][1]:
        assert rows[i][2] < rows[i+1][2], f"Tie-break fail at ranks {rows[i][0]},{rows[i+1][0]}"

sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]
hp = sum(1 for _, r in top_feats.iterrows()
         if is_honeypot(CandidateFeatures(**{f.name: r[f.name] for f in fields(CandidateFeatures) if f.name in r.index}, profile_text='')))
print(f"Rows: {len(df)} | Scores: {scores[0]:.4f} → {scores[-1]:.4f} | Honeypots: {hp}")
assert hp == 0
print("All checks passed.")

Rows: 100 | Scores: 1.0000 → 0.7680 | Honeypots: 0
All checks passed.


In [6]:
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

,rank,candidate_id,score,reasoning
0,1,CAND_0018549,1.0000,Recommendation Systems Engineer at Uber with 6+ years building ranking and r...
1,2,CAND_0047721,1.0000,Senior Data Scientist at Microsoft with 7+ years building ranking and retrie...
2,3,CAND_0062247,1.0000,AI Engineer at Google with 7+ years building ranking and retrieval systems a...
3,4,CAND_0081846,1.0000,Lead AI Engineer at Razorpay with 6+ years building ranking and retrieval sy...
4,5,CAND_0077337,0.9984,Staff Machine Learning Engineer at Paytm with 6+ years building ranking and ...
5,6,CAND_0092278,0.9976,Senior NLP Engineer at Microsoft with 6+ years building ranking and retrieva...
6,7,CAND_0087630,0.9974,AI Engineer at Vedantu with 7+ years building ranking and retrieval systems ...
7,8,CAND_0039383,0.9969,Applied ML Engineer at Meesho with 7+ years building ranking and retrieval s...
8,9,CAND_0044222,0.9969,AI Engineer at PolicyBazaar with 7+ years building ranking and retrieval sys...
9,10,CAND_0018499,0.9968,Senior Machine Learning Engineer at Zomato with 7+ years building ranking an...


In [8]:
!python tests/validate_submission.py "H:\india_runs\jashwanth_s.csv"

Submission is valid.


In [ ]:
from IPython.display import FileLink
FileLink('jashwanth_s.csv')